In [ ]:
#Save Safe Dataset
import os

os.makedirs("safe_output", exist_ok=True)

df.to_csv(
    "safe_output/transactions_safe.csv",
    index=False
)

print("Safe dataset saved successfully!")

Safe dataset saved successfully!


In [ ]:
#see what the raw PII lookup table looks like and what the safe dataset looks like
print("Raw PII Lookup Table:")
print(pii_lookup.head())

print("\nSafe Dataset:")
print(df.head())

Raw PII Lookup Table:
                                 txn_id            name          nic  \
0  9a73daed-3bf0-4708-9dd5-91ec134831b5  Ashley Roberts  242-79-8743   
1  4f5d4fa8-7388-49ea-ac27-22e107d3bf8d    Kirsten King  213-27-7129   
2  c75d56c2-de37-4580-aca1-12aff7f45cab       Eric Wood  225-66-4625   
3  ad1be3ac-7210-4fcc-aa4b-ed733010e578     Ryan Weaver  676-75-2563   
4  01db86a2-9871-4faf-88b7-ed3227e1fe9d   Sarah Johnson  163-76-0503   

           account_no  
0  RQGK15708015028536  
1  HYRC49073441955416  
2  XHCM31516952259079  
3  MWBO37396375351058  
4  AQXK60933373591022  

Safe Dataset:
                                 txn_id    amount txn_type  \
0  9a73daed-3bf0-4708-9dd5-91ec134831b5  32007.40   credit   
1  4f5d4fa8-7388-49ea-ac27-22e107d3bf8d  37103.37   credit   
2  c75d56c2-de37-4580-aca1-12aff7f45cab  11238.22   credit   
3  ad1be3ac-7210-4fcc-aa4b-ed733010e578  33867.30   credit   
4  01db86a2-9871-4faf-88b7-ed3227e1fe9d  29565.58   credit   

             

In [ ]:
#PII Tokenization
import hashlib

# These are your PII columns — ONLY these get hashed
PII_COLUMNS = ["name", "nic", "account_no"]

def sha256_hash(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

# Save the PII lookup table ON LAPTOP — never uploaded
pii_lookup = df[["txn_id"] + PII_COLUMNS].copy()
pii_lookup.to_csv("pii_lookup_table.csv", index=False)
print("PII lookup saved locally — DO NOT upload this file")

# Now hash the PII columns in the main dataframe
for col in PII_COLUMNS:
    df[col + "_token"] = df[col].apply(sha256_hash)
    df.drop(columns=[col], inplace=True)

print(df.columns.tolist())
print(df.head(2))

KeyError: "['name', 'nic', 'account_no'] not in index"

In [11]:
#Generate Raw Banking Data
from faker import Faker
import pandas as pd
import random

fake = Faker()
random.seed(42)

rows = []
for _ in range(10000):
    rows.append({
        "txn_id":       fake.uuid4(),
        "name":         fake.name(),           # PII
        "nic":          fake.ssn(),            # PII
        "account_no":   fake.bban(),           # PII
        "amount":       round(random.uniform(100, 50000), 2),
        "txn_type":     random.choice(["credit","debit"]),
        "timestamp":    str(fake.date_time_this_year()),
        "branch":       fake.city()
    })

df = pd.DataFrame(rows)
print(df.head(3))

                                 txn_id             name          nic  \
0  e4af6317-7826-4845-ba1a-eddd6e6e018c      Erica Perez  713-27-4018   
1  b0bcebc2-8bef-4d4c-a0d0-82d5dbdec8a9     Joseph Clark  212-15-9272   
2  594cb695-4815-487e-bb1a-4cabb8d71dd8  Ashley Mckinney  414-35-1906   

           account_no    amount txn_type            timestamp          branch  
0  JJHP73560730737228  32007.40   credit  2026-05-17 09:10:44  West Reneefort  
1  JHKQ62590215238564  37103.37   credit  2026-02-24 05:57:33        Amymouth  
2  BDFG81885691272568  11238.22   credit  2026-04-09 14:35:48      Thomasside  
